In [ ]:
import objaverse

uids = objaverse.load_uids()
print(len(uids))

In [ ]:
import random
random.seed(42)

random_ojbect_uids = random.sample(uids, 100)

In [ ]:
objects = objaverse.load_objects(random_ojbect_uids)

In [ ]:
import torch
from pytorch3d.io import IO
from pytorch3d.io.experimental_gltf_io import MeshGlbFormat
all_keys = list(objects.keys())
key1 = all_keys[0]
obj1_path = objects[key1]
device = 'cuda'
io = IO(include_default_formats=False)
io.register_meshes_format(MeshGlbFormat())
obj1 = io.load_mesh(obj1_path, device=device, include_textures=True)

In [ ]:
# rotate the object
from pytorch3d.transforms import Transform3d
x_neg_90 = Transform3d(device=device).rotate_axis_angle(angle=180, axis="x", degrees=True)
obj1 = obj1.update_padded(
    x_neg_90.transform_points(obj1.verts_padded()),
)

In [ ]:
from pytorch3d.io import load_objs_as_meshes

obj1 = load_objs_as_meshes(['demo_data/can/f82039689f504922995936c68484aa61.obj'])

FileNotFoundError: [Errno 2] No such file or directory: 'd'

In [ ]:
from pytorch3d.structures import Meshes
from pytorch3d.renderer import TexturesUV, look_at_rotation, FoVPerspectiveCameras, MeshRenderer, MeshRasterizer
from pytorch3d.renderer import RasterizationSettings, PointLights
from pytorch3d.renderer.blending import BlendParams
from pano_utils import cubemap_to_equirectangular, pano_to_pointcloud, DistShader, MultiHitHardPhongShader, NormalAngleShader, get_sampling_prob_from_normal_angle

# Define view directions
camera_dirs = torch.tensor([
    [-1, 0, 0],   # -X
    [1, 0, 0],    # +X
    [0, -1, 0],   # -Y
    [0, 1, 0],    # +Y
    [0, 0, -1],   # -Z
    [0, 0, 1],    # +Z
], device=device, dtype=torch.float32)

# Custom up vectors to avoid singularities
up_vectors = torch.tensor([
    [0, -1, 0],    # for -X
    [0, 1, 0],    # for +X
    [0, 0, -1],   # for -Y
    [0, 0, 1],    # for +Y (must not be parallel to +Y dir)
    [0, -1, 0],    # for -Z
    [0, 1, 0],    # for +Z
], device=device, dtype=torch.float32)

# Compute rotation matrices
R = look_at_rotation(camera_dirs, up=up_vectors)
T = torch.zeros_like(camera_dirs)  # Camera is at the origin

def mesh_to_pano(mesh, image_size=512, max_hits=5):
    verts = mesh.verts_packed()
    r2 = verts.norm(dim=1, keepdim=True) ** 2
    inverted_verts = verts / r2

    faces = mesh.faces_packed()
    flipped_faces = faces[:, [0, 2, 1]]
    verts_uvs = mesh.textures.verts_uvs_padded()  # (1, V, 2)
    faces_uvs = mesh.textures.faces_uvs_padded()  # (1, F, 3)
    flipped_fuvs = faces_uvs[..., [0, 2, 1]]

    new_textures = TexturesUV(
        maps=mesh.textures._maps_padded,
        faces_uvs=flipped_fuvs.to(device),  # (1, F, 3)
        verts_uvs=verts_uvs.to(device),  # (1, V, 2)
    )

    inverted_mesh = Meshes(verts=[inverted_verts], faces=[flipped_faces], textures=new_textures)


    cameras = FoVPerspectiveCameras(device=device, R=R, T=T, fov=90, znear=0.1, zfar=200)

    # Set up renderer
    raster_settings = RasterizationSettings(image_size=image_size, blur_radius=0.0, faces_per_pixel=max_hits)
    lights = PointLights(device=device, location=[[0, 0, 0.]])
    renderer = MeshRenderer(
        rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
        shader=MultiHitHardPhongShader(device=device, cameras=cameras, lights=lights, max_hits=max_hits)
    )
    blend_params = BlendParams(background_color=(-1, -1, -1))

    # Render images
    all_images = renderer(inverted_mesh.extend(6), cameras=cameras, lights=lights, blend_params=blend_params)  # (6, H, W, 3)faces.shape

    all_panos = []
    pano_valid_regions = []
    for images in all_images:
        cube_images = images[..., :3].permute(0, 3, 1, 2)  # (6, C, H, W)
        cube_images[0] = torch.flip(cube_images[0], dims=(2,))  # Flip -X face horizontally
        this_pano = cubemap_to_equirectangular(cube_images, H=image_size, device=device)
        all_panos.append(this_pano)
        this_pano_valid_region = this_pano.mean(axis=0) > 0
        pano_valid_regions.append(this_pano_valid_region)

    dist_renderer = MeshRenderer(
        rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
        shader=DistShader(cameras=cameras, max_hits=max_hits)
    )
    dist_maps = dist_renderer(inverted_mesh.extend(6), cameras=cameras)  # (6, H, W, 1)
    all_pano_dist = []
    for this_dist_maps in dist_maps:
        this_dist_maps = this_dist_maps.squeeze(-1).permute(0, 1, 2)  # (6, H, W)
        this_dist_maps[0] = torch.flip(this_dist_maps[0], dims=(1,))  # Flip -X face horizontally
        this_pano_dist = cubemap_to_equirectangular(
            this_dist_maps[:, None],  # 6, H, W -> 6, 1, H, W
            H=image_size, 
            device=device
        )[0]
        all_pano_dist.append(this_pano_dist)

    normal_angle_renderer = MeshRenderer(
        rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
        shader=NormalAngleShader(cameras=cameras, max_hits=max_hits)
    )
    normal_angle_maps = normal_angle_renderer(inverted_mesh.extend(6), cameras=cameras)  # (6, H, W, 1)
    all_pano_normal_angle = []
    for this_normal_angle_maps in normal_angle_maps:
        this_normal_angle_maps = this_normal_angle_maps.squeeze(-1).permute(0, 1, 2)  # (6, 3, H)
        this_normal_angle_maps[0] = torch.flip(this_normal_angle_maps[0], dims=(1,))  # Flip -X face horizontally
        this_pano_normal_angle = cubemap_to_equirectangular(
            this_normal_angle_maps[:, None],  # 6, H, W -> 6, 1, H, W
            H=image_size,
            device=device
        )[0]
        all_pano_normal_angle.append(this_pano_normal_angle)

    return all_panos, pano_valid_regions, all_pano_dist, all_pano_normal_angle

In [ ]:
all_panos, pano_valid_regions, all_pano_dist, all_pano_normal_angle = mesh_to_pano(obj1, image_size=512, max_hits=5)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.imshow(all_panos[0].cpu().numpy().transpose(1, 2, 0))

In [ ]:
def pano_to_mesh_3d(pano_color, pano_dist, valid_mask=None, max_edge_ratio=1.5):
    """
    Convert layered panorama + distance maps into a 3D mesh by triangulating both
    intra-layer (xy plane) and inter-layer (z direction) neighbors.

    Args:
        pano_color: (L, 3, H, 2H) RGB panoramas
        pano_dist:  (L, H, 2H)    Distance-to-origin maps
        valid_mask: (L, H, 2H)    Optional validity masks
        max_edge_ratio: float     Reject triangles with long edges (relative to median)

    Returns:
        verts:  (N, 3) FloatTensor
        faces:  (M, 3) LongTensor
        colors: (N, 3) FloatTensor
    """
    L, C, H, W = pano_color.shape
    device = pano_color.device

    pano_color = pano_color.permute(0, 2, 3, 1)  # (L, H, W, 3)
    pano_dist = pano_dist                       # (L, H, W)

    if valid_mask is None:
        valid_mask = torch.isfinite(pano_dist) & (pano_dist > 1e-5)

    # Step 1: Compute direction map (spherical coordinates)
    phi = torch.linspace(0, np.pi, H, device=device).view(H, 1)
    theta = torch.linspace(-np.pi, np.pi, W, device=device).view(1, W)
    x = torch.sin(phi) * torch.sin(theta)
    y = torch.cos(phi).repeat(1, W)
    z = torch.sin(phi) * torch.cos(theta)
    dirs = torch.stack([x, y, z], dim=-1)  # (H, W, 3)

    dirs = dirs.unsqueeze(0).expand(L, -1, -1, -1)  # (L, H, W, 3)
    pts = dirs * pano_dist.unsqueeze(-1)           # (L, H, W, 3)

    # Step 2: flatten valid pixels to verts/colors
    index_map = -torch.ones((L, H, W), dtype=torch.long, device=device)
    valid_idx = valid_mask.nonzero(as_tuple=False)
    index_map[valid_mask] = torch.arange(valid_idx.shape[0], device=device)

    verts = pts[valid_mask]                 # (N, 3)
    colors = pano_color[valid_mask]         # (N, 3)

    faces = []

    # Helper: construct triangles between 4 corners (a quad) using 2 triangles
    def tri_from_quad(i00, i01, i10, i11):
        m_a = (i00 >= 0) & (i01 >= 0) & (i10 >= 0)
        m_b = (i11 >= 0) & (i01 >= 0) & (i10 >= 0)
        tri_a = torch.stack([i00[m_a], i01[m_a], i10[m_a]], dim=-1)
        tri_b = torch.stack([i11[m_b], i01[m_b], i10[m_b]], dim=-1)
        return [tri_a, tri_b]

    # Step 3: in-layer XY triangles
    y = torch.arange(H - 1, device=device)
    x = torch.arange(W, device=device)
    yy, xx = torch.meshgrid(y, x, indexing='ij')

    for l in range(L):
        i00 = index_map[l, yy, xx]
        i01 = index_map[l, yy, (xx + 1) % W]
        i10 = index_map[l, yy + 1, xx]
        i11 = index_map[l, yy + 1, (xx + 1) % W]
        faces += tri_from_quad(i00, i01, i10, i11)

    # Step 4: inter-layer Z triangles (connect adjacent depth layers)
    z = torch.arange(L - 1, device=device)
    y = torch.arange(H - 1, device=device)
    x = torch.arange(W, device=device)
    zz, yy, xx = torch.meshgrid(z, y, x, indexing='ij')

    i0 = index_map[zz, yy, xx]
    i1 = index_map[zz + 1, yy, xx]
    for dx, dy in [(1, 0), (0, 1), (-1, 0), (0, -1)]: #, (1, 1), (-1, -1), (1, -1), (-1, 1)]:
        i2 = index_map[zz + 1, yy + dy, (xx + dx) % W]

        mask1 = (i0 >= 0) & (i1 >= 0) & (i2 >= 0)
        tri1 = torch.stack([i0[mask1], i1[mask1], i2[mask1]], dim=-1)
        faces += [tri1]

    faces = torch.cat(faces, dim=0)  # (M, 3)

    # Step 5: optional filtering of long-edge triangles
    v0, v1, v2 = verts[faces[:, 0]], verts[faces[:, 1]], verts[faces[:, 2]]
    e1 = (v0 - v1).norm(dim=1)
    e2 = (v1 - v2).norm(dim=1)
    e3 = (v2 - v0).norm(dim=1)
    edge_lengths = torch.cat([e1, e2, e3])
    edge_median = edge_lengths.median()
    valid_tri = (e1 < edge_median * max_edge_ratio) & \
                (e2 < edge_median * max_edge_ratio) & \
                (e3 < edge_median * max_edge_ratio)

    faces = faces[valid_tri]

    return verts, faces, colors

all_verts, all_faces, all_colors = pano_to_mesh_3d(
    torch.stack(all_panos, dim=0),  # (L, 3, H, 2H)
    torch.stack(all_pano_dist, dim=0),  # (L, H, 2H)
    valid_mask=torch.stack(pano_valid_regions, dim=0),  # (L, H, 2H)
    max_edge_ratio=3
)
all_verts = all_verts / (all_verts.norm(dim=1, keepdim=True) ** 2)


In [ ]:
import open3d as o3d

mesh = o3d.geometry.TriangleMesh()
mesh.vertices = o3d.utility.Vector3dVector(all_verts.cpu().numpy())
mesh.triangles = o3d.utility.Vector3iVector(all_faces.cpu().numpy())
mesh.vertex_colors = o3d.utility.Vector3dVector(all_colors.cpu().numpy())
mesh.compute_vertex_normals()

# mesh = mesh.filter_smooth_simple(number_of_iterations=5)
# mesh.remove_unreferenced_vertices()
# mesh.remove_degenerate_triangles()
# mesh.remove_duplicated_triangles()
# mesh = mesh.simplify_quadric_decimation(target_number_of_triangles=100_000)

In [ ]:
# Step 1: Create point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(all_verts.cpu().numpy())
pcd.colors = o3d.utility.Vector3dVector(all_colors.cpu().numpy())

pcd = pcd.voxel_down_sample(0.005)
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=12, std_ratio=4.0)

# Step 2: Estimate normals
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.02, max_nn=50)
)
pcd.orient_normals_consistent_tangent_plane(k=100)

# Step 3: Poisson reconstruction
mesh2, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(densities)
# mesh2.remove_vertices_by_mask(dens < np.quantile(dens, 0.01))

# Step 4: Optional: crop low-density noise
bbox = pcd.get_axis_aligned_bounding_box()
mesh2 = mesh2.crop(bbox)

In [ ]:
from copy import deepcopy
pcd_tree = o3d.geometry.KDTreeFlann(pcd)
mesh_verts = np.asarray(mesh2.vertices)
mask = []

for v in mesh_verts:
    [_, _, d2] = pcd_tree.search_knn_vector_3d(v, 1)
    mask.append(np.sqrt(d2[0]) < 0.2)

mask = np.array(mask)
mesh2_cp = deepcopy(mesh2)
mesh2_cp.remove_vertices_by_mask(~mask)

In [ ]:
import numpy as np
from pythreejs import *
from IPython.display import display

mesh_to_show = mesh

verts  = np.asarray(mesh_to_show.vertices,  dtype=np.float32)        # (V, 3)
faces  = np.asarray(mesh_to_show.triangles, dtype=np.uint32)         # (F, 3)
vcolors = np.asarray(mesh_to_show.vertex_colors, dtype=np.float32)

geometry = BufferGeometry(
    attributes={
        'position': BufferAttribute(verts,  normalized=False),
        'color'   : BufferAttribute(vcolors, normalized=False),
    },
    index=BufferAttribute(faces.flatten(), normalized=False)
)

material = MeshLambertMaterial(vertexColors='VertexColors',
                               side='DoubleSide')   # show both sides

mesh_obj = Mesh(geometry=geometry, material=material)

camera = PerspectiveCamera(position=[2, 2, 2], up=[0, 0, 1])
camera.lookAt([0, 0, 0])

lights = [
    AmbientLight(intensity=1.0),
    DirectionalLight(position=[4, 4, 6], intensity=0.7),
]

scene = Scene(children=[mesh_obj, camera, *lights])

controls  = OrbitControls(controlling=camera)
renderer  = Renderer(camera=camera, scene=scene,
                     controls=[controls], width=600, height=600)

display(renderer)